<a href="https://colab.research.google.com/github/adhishmodi/customer-churn-prediction/blob/main/notebooks/09_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/adhishmodi/customer-churn-prediction/blob/main/notebooks/09_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Milestone 5 — Phase 2B: Feature Engineering Experiments

Compare the original feature set against three deterministic engineered variants: tenure groups, subscribed-service count, and both together. Compare the three retained Phase 2A finalists on identical 5-fold × 3-repeat stratified CV splits.

**Leakage rule:** only the reconstructed training partition is passed to experiment functions. The old held-out test split is discarded and must not be used for feature selection or model decisions. Every transformation is inside the sklearn pipeline.


## 1. Environment and dataset setup

Run from the repository root or in Google Colab. The raw CSV is not committed to Git; upload the original IBM Telco Customer Churn CSV when prompted.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = "https://github.com/adhishmodi/customer-churn-prediction.git"
EXPECTED_CSV = "WA_Fn-UseC_-Telco-Customer-Churn.csv"
current_dir = Path.cwd()
PROJECT_ROOT = next(
    (p for p in (current_dir, *current_dir.parents)
     if (p / "src").is_dir() and (p / "notebooks").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path("/content/customer-churn-prediction")
    if not (PROJECT_ROOT / "src").is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch",
             "fix/phase-2b-feature-routing",
             REPOSITORY_URL, str(PROJECT_ROOT)], check=True,
        )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
DATA_PATH = PROJECT_ROOT / "data" / "raw" / EXPECTED_CSV
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Expected dataset: {DATA_PATH}")


In [ ]:
if not DATA_PATH.exists():
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(
            f"Dataset not found at {DATA_PATH}. Place the original CSV there."
        ) from exc
    print(f"Upload the original file named {EXPECTED_CSV}")
    uploaded = files.upload()
    csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
    if EXPECTED_CSV in uploaded:
        selected_name = EXPECTED_CSV
    elif len(csv_names) == 1:
        selected_name = csv_names[0]
    else:
        raise ValueError(
            f"Upload exactly one CSV, preferably named {EXPECTED_CSV}. Found: {csv_names}"
        )
    DATA_PATH.write_bytes(uploaded[selected_name])
assert DATA_PATH.exists(), "Dataset setup did not create the expected CSV file."


## 2. Reconstruct training data only

Recreate the original stratified split (seed 42). The holdout is immediately deleted. It is never supplied to the experiment function.


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

from src.preprocess import load_and_clean_data, prepare_features_target, split_data
from src.train import get_imbalance_experiment_models
from src.experiments import compare_feature_engineering_repeated_cv
from src.feature_engineering import FEATURE_CONFIGS

cleaned_df = load_and_clean_data(DATA_PATH)
X, y = prepare_features_target(cleaned_df)
X_train, held_out_features, y_train, held_out_labels = split_data(X, y)
del held_out_features, held_out_labels, X, y, cleaned_df

print(f"Training rows: {len(X_train):,}")
print(f"Training churn prevalence: {y_train.mean():.2%}")
print("Held-out test set used: False")


## 3. Define controlled feature configurations

- `original`: existing 19 predictors, no additions.
- `tenure_groups`: add tenure bands (0–12, 13–24, 25–48, 49–60, 61+ months).
- `service_count`: count `Yes` across eight binary service flags: phone service, multiple lines, online security, online backup, device protection, tech support, streaming TV and streaming movies. `No internet service` and `No phone service` are not counted.
- `tenure_groups_and_service_count`: add both features.

The original predictors remain in every configuration. These transformations are deterministic and learn no global statistics; they still execute inside each CV pipeline.


In [ ]:
candidate_models_all = get_imbalance_experiment_models()
CANDIDATE_NAMES = [
    "XGBoost (unweighted)",
    "XGBoost (balanced)",
    "Random Forest (balanced)",
]
candidate_models = {name: candidate_models_all[name] for name in CANDIDATE_NAMES}
FEATURE_CONFIG_NAMES = list(FEATURE_CONFIGS)
N_SPLITS = 5
N_REPEATS = 3
RANDOM_STATE = 42

print("Models:", list(candidate_models))
print("Feature configurations:", FEATURE_CONFIG_NAMES)
print(f"Validation: {N_SPLITS} folds × {N_REPEATS} repeats")
print("Total candidate configurations:", len(candidate_models) * len(FEATURE_CONFIG_NAMES))


## 4. Feature-routing preflight

Before repeated CV, fit a lightweight dummy pipeline for each configuration and assert that the engineered columns appear in the preprocessor output. This prevents a `ColumnTransformer` from silently dropping engineered features.


In [ ]:
from sklearn.dummy import DummyClassifier
from src.train import build_model_pipeline

feature_routing_audit = {}
for feature_config in FEATURE_CONFIG_NAMES:
    probe = build_model_pipeline(
        DummyClassifier(strategy="prior"), feature_config=feature_config
    )
    probe.fit(X_train, y_train)
    encoded_names = probe.named_steps["preprocessor"].get_feature_names_out()
    has_tenure_group = any(
        str(name).startswith("tenure_group_") for name in encoded_names
    )
    has_service_count = "service_count" in encoded_names
    expects_tenure_group = FEATURE_CONFIGS[feature_config]["tenure_groups"]
    expects_service_count = FEATURE_CONFIGS[feature_config]["service_count"]
    assert has_tenure_group == expects_tenure_group, feature_config
    assert has_service_count == expects_service_count, feature_config
    feature_routing_audit[feature_config] = {
        "tenure_group_encoded": has_tenure_group,
        "service_count_included": has_service_count,
        "classifier_input_features": int(probe.named_steps["classifier"].n_features_in_),
    }

print("Feature-routing audit passed:")
for feature_config, details in feature_routing_audit.items():
    print(feature_config, details)


## 5. Run repeated stratified cross-validation

All 12 model/feature combinations receive the same 15 train/validation splits. Preprocessing and feature engineering are fitted/applied within each pipeline. This may take several minutes in Colab.


In [ ]:
feature_results = compare_feature_engineering_repeated_cv(
    X_train,
    y_train,
    models=candidate_models,
    feature_configs=FEATURE_CONFIG_NAMES,
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    random_state=RANDOM_STATE,
)
assert feature_results["configuration"]["test_set_used"] is False
assert len(feature_results["fold_scores"]) == (
    len(candidate_models) * len(FEATURE_CONFIG_NAMES) * N_SPLITS * N_REPEATS
)
print(feature_results["configuration"])
print("Phase 2B repeated-CV experiment completed.")


## 6. Compare results

ROC-AUC is the primary ranking metric, with Average Precision as a second ranking measure. Inspect precision, recall, F1, accuracy and fit time before interpreting trade-offs. Repeated-fold standard deviations are descriptive, not confidence intervals or significance tests.


In [ ]:
summary = feature_results["summary"].copy()
display_columns = [
    "model", "feature_config", "fold_count",
    "roc_auc_mean", "roc_auc_std",
    "average_precision_mean", "average_precision_std",
    "precision_mean", "recall_mean", "f1_mean", "accuracy_mean", "fit_time_mean",
]
display(summary[display_columns].sort_values("roc_auc_mean", ascending=False).round(4))

pivot_auc = summary.pivot(index="model", columns="feature_config", values="roc_auc_mean")
pivot_ap = summary.pivot(index="model", columns="feature_config", values="average_precision_mean")
print("Mean ROC-AUC by model and feature configuration")
display(pivot_auc.round(4))
print("Mean Average Precision by model and feature configuration")
display(pivot_ap.round(4))


In [ ]:
display(feature_results["repeat_winners"])
display(feature_results["selection_stability"])

print("Interpretation checklist:")
print("1. Does either engineered feature improve ROC-AUC and Average Precision consistently?")
print("2. Are improvements consistent across all three model families?")
print("3. Does service count duplicate information already represented by the original flags?")
print("4. Does tenure grouping help beyond the original numeric tenure feature?")
print("5. Are any gains large relative to observed fold variability and fit cost?")
print("6. Do not inspect or use the historical holdout for these decisions.")


## 7. Save training-only outputs

CSV files are written to the ignored `reports/` directory. Upload the four Phase 2B CSVs for review before recording conclusions in the roadmap. Do not report a feature configuration as superior based on a single fold or one metric.


In [ ]:
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
outputs = {
    "phase_2b_feature_engineering_fold_scores.csv": feature_results["fold_scores"],
    "phase_2b_feature_engineering_summary.csv": summary,
    "phase_2b_feature_engineering_repeat_winners.csv": feature_results["repeat_winners"],
    "phase_2b_feature_engineering_selection_stability.csv": feature_results["selection_stability"],
}
for filename, frame in outputs.items():
    frame.to_csv(REPORTS_DIR / filename, index=False)
    print(f"Saved: {REPORTS_DIR / filename}")


## 8. Protocol assertions

Confirm all candidates, folds and metrics completed successfully. The final holdout remains unused.


In [ ]:
expected_rows = len(candidate_models) * len(FEATURE_CONFIG_NAMES) * N_SPLITS * N_REPEATS
assert len(feature_results["fold_scores"]) == expected_rows == 180
metric_columns = [
    "roc_auc", "average_precision", "precision", "recall", "f1", "accuracy"
]
assert np.isfinite(feature_results["fold_scores"][metric_columns].to_numpy()).all()
assert feature_results["configuration"]["test_set_used"] is False
assert feature_results["configuration"]["candidate_count"] == 12
assert set(feature_results["fold_scores"]["feature_config"]) == set(FEATURE_CONFIG_NAMES)
print("All Phase 2B protocol assertions passed.")
print("Training-only CV rows evaluated:", len(feature_results["fold_scores"]))
print("Held-out test set used: False")


## Completion checklist

- [ ] Original CSV uploaded and validated.
- [ ] All 12 configurations completed on 5-fold × 3-repeat training-only CV.
- [ ] All four Phase 2B CSV outputs saved and reviewed.
- [ ] Feature-engineering conclusions recorded in `PROJECT_STATUS.md` and the final report.

Do not check these items until the notebook has been run and its outputs reviewed.
